# Checkpoints


In [123]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

source_str = "orders"
sink_str = "orders_aggregated"

tn = (
    Tn.source(source_str)
    .map(lambda r: r["value"])
    .group_by_agg(lambda r: r["customer_id"],
                  lambda r: r,
                  lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                                    "product_ids": sorted(agg_r["product_ids"] + [r["product_id"]])},
                  {"orders": 0, "product_ids": []},
                  lambda by, agg_r: {"customer_id": by,
                                     "orders": agg_r["orders"],
                                     "product_ids": agg_r["product_ids"]})
    .map(lambda r: {"key": r["customer_id"],
                    "value": r})
    .sink(sink_str)
)

built_tn = Tn.build(tn)



In [ ]:
import random

class OrderGenerator:
    def __init__(self):
        self.order_id_int = 0
        self.customer_id_int = 0
        #
        self.ts_int = 0
        self.ts_step_int = 1

    def generate(self):
        m = {
            "key": self.order_id_int,
            "value": {"id": self.order_id_int,
                      "product_id": random.randint(0, 100 - 1),
                      "customer_id": random.randint(0, 10 - 1),
                      "ts": self.ts_int},
        }
        #
        self.order_id_int += 1
        #
        self.ts_int += self.ts_step_int
        #
        return m

#

gen = OrderGenerator()
for _ in range(3):
    print(gen.generate())


{'key': 0, 'value': {'id': 0, 'product_id': 35, 'customer_id': 0, 'ts': 0}}
{'key': 1, 'value': {'id': 1, 'product_id': 55, 'customer_id': 5, 'ts': 1}}
{'key': 2, 'value': {'id': 2, 'product_id': 21, 'customer_id': 4, 'ts': 2}}


In [65]:
built_tn.reset()
gen = OrderGenerator()
source_m_list = []
sink_m_list = []
for _ in range(10):
    m = gen.generate()
    source_m_list += [m]
    built_tn.push(source_str, [m])
    sink_m_list += built_tn.latest()[sink_str]

#

source_key_int_value_dict_dict = {}
for m in source_m_list:
    order_id_int = m["value"]["id"]
    product_id_int = m["value"]["product_id"]
    customer_id_int = m["value"]["customer_id"]
    #
    agg_orders_int = source_key_int_value_dict_dict.get(customer_id_int, {}).get("orders", 0)
    agg_product_ids_int_list = source_key_int_value_dict_dict.get(customer_id_int, {}).get("product_ids", [])
    #
    source_key_int_value_dict_dict[customer_id_int] = {"customer_id": customer_id_int,
                                                       "orders": agg_orders_int + 1,
                                                       "product_ids": sorted(agg_product_ids_int_list + [product_id_int])}

#

sink_key_int_value_dict_dict = {}
for m in sink_m_list:
    sink_key_int_value_dict_dict[m["key"]] = m["value"]

#

print(source_key_int_value_dict_dict)
print(sink_key_int_value_dict_dict)

if source_key_int_value_dict_dict != sink_key_int_value_dict_dict:
    raise Exception("Test failed.")

#

print("Test successful.")


{9: {'customer_id': 9, 'orders': 1, 'product_ids': [76]}, 7: {'customer_id': 7, 'orders': 2, 'product_ids': [52, 64]}, 8: {'customer_id': 8, 'orders': 1, 'product_ids': [28]}, 5: {'customer_id': 5, 'orders': 2, 'product_ids': [28, 67]}, 1: {'customer_id': 1, 'orders': 1, 'product_ids': [6]}, 4: {'customer_id': 4, 'orders': 1, 'product_ids': [24]}, 0: {'customer_id': 0, 'orders': 2, 'product_ids': [38, 90]}}
{9: {'customer_id': 9, 'orders': 1, 'product_ids': [76]}, 7: {'customer_id': 7, 'orders': 2, 'product_ids': [52, 64]}, 8: {'customer_id': 8, 'orders': 1, 'product_ids': [28]}, 5: {'customer_id': 5, 'orders': 2, 'product_ids': [28, 67]}, 1: {'customer_id': 1, 'orders': 1, 'product_ids': [6]}, 4: {'customer_id': 4, 'orders': 1, 'product_ids': [24]}, 0: {'customer_id': 0, 'orders': 2, 'product_ids': [38, 90]}}
Test successful.


In [66]:
gen = OrderGenerator()


#

built_tn.reset()
source_m_list = []
sink_m_list = []
for _ in range(10):
    m = gen.generate()
    source_m_list += [m]
    built_tn.push(source_str, [m])
    sink_m_list += built_tn.latest()[sink_str]

#

for _ in range(10):
    m = gen.generate()
    source_m_list += [m]
    built_tn.push(source_str, [m])
    sink_m_list += built_tn.latest()[sink_str]

#

source_key_int_value_dict_dict = {}
for m in source_m_list:
    order_id_int = m["value"]["id"]
    product_id_int = m["value"]["product_id"]
    customer_id_int = m["value"]["customer_id"]
    #
    agg_orders_int = source_key_int_value_dict_dict.get(customer_id_int, {}).get("orders", 0)
    agg_product_ids_int_list = source_key_int_value_dict_dict.get(customer_id_int, {}).get("product_ids", [])
    #
    source_key_int_value_dict_dict[customer_id_int] = {"customer_id": customer_id_int,
                                                       "orders": agg_orders_int + 1,
                                                       "product_ids": sorted(agg_product_ids_int_list + [product_id_int])}

#

sink_key_int_value_dict_dict = {}
for m in sink_m_list:
    sink_key_int_value_dict_dict[m["key"]] = m["value"]

#

print(source_key_int_value_dict_dict)
print(sink_key_int_value_dict_dict)

if source_key_int_value_dict_dict != sink_key_int_value_dict_dict:
    raise Exception("Test failed.")

#

print("Test successful.")


{2: {'customer_id': 2, 'orders': 2, 'product_ids': [20, 58]}, 1: {'customer_id': 1, 'orders': 3, 'product_ids': [27, 58, 62]}, 8: {'customer_id': 8, 'orders': 4, 'product_ids': [31, 36, 55, 91]}, 6: {'customer_id': 6, 'orders': 3, 'product_ids': [14, 49, 60]}, 5: {'customer_id': 5, 'orders': 3, 'product_ids': [13, 95, 95]}, 0: {'customer_id': 0, 'orders': 2, 'product_ids': [36, 89]}, 9: {'customer_id': 9, 'orders': 1, 'product_ids': [67]}, 7: {'customer_id': 7, 'orders': 1, 'product_ids': [47]}, 3: {'customer_id': 3, 'orders': 1, 'product_ids': [15]}}
{2: {'customer_id': 2, 'orders': 2, 'product_ids': [20, 58]}, 1: {'customer_id': 1, 'orders': 3, 'product_ids': [27, 58, 62]}, 8: {'customer_id': 8, 'orders': 4, 'product_ids': [31, 36, 55, 91]}, 6: {'customer_id': 6, 'orders': 3, 'product_ids': [14, 49, 60]}, 5: {'customer_id': 5, 'orders': 3, 'product_ids': [13, 95, 95]}, 0: {'customer_id': 0, 'orders': 2, 'product_ids': [36, 89]}, 9: {'customer_id': 9, 'orders': 1, 'product_ids': [67]}

In [67]:
import cloudpickle

#

gen = OrderGenerator()


#

built_tn.reset()
source_m_list = []
sink_m_list = []
for _ in range(10):
    m = gen.generate()
    source_m_list += [m]
    built_tn.push(source_str, [m])
    sink_m_list += built_tn.latest()[sink_str]

#

x = cloudpickle.dumps(built_tn)

built_tn.reset()

built_tn = cloudpickle.loads(x)

#

for _ in range(10):
    m = gen.generate()
    source_m_list += [m]
    built_tn.push(source_str, [m])
    sink_m_list += built_tn.latest()[sink_str]

#

source_key_int_value_dict_dict = {}
for m in source_m_list:
    order_id_int = m["value"]["id"]
    product_id_int = m["value"]["product_id"]
    customer_id_int = m["value"]["customer_id"]
    #
    agg_orders_int = source_key_int_value_dict_dict.get(customer_id_int, {}).get("orders", 0)
    agg_product_ids_int_list = source_key_int_value_dict_dict.get(customer_id_int, {}).get("product_ids", [])
    #
    source_key_int_value_dict_dict[customer_id_int] = {"customer_id": customer_id_int,
                                                       "orders": agg_orders_int + 1,
                                                       "product_ids": sorted(agg_product_ids_int_list + [product_id_int])}

#

sink_key_int_value_dict_dict = {}
for m in sink_m_list:
    sink_key_int_value_dict_dict[m["key"]] = m["value"]

#

print(source_key_int_value_dict_dict)
print(sink_key_int_value_dict_dict)

if source_key_int_value_dict_dict != sink_key_int_value_dict_dict:
    raise Exception("Test failed.")

#

print("Test successful.")


{5: {'customer_id': 5, 'orders': 1, 'product_ids': [75]}, 7: {'customer_id': 7, 'orders': 2, 'product_ids': [35, 44]}, 3: {'customer_id': 3, 'orders': 4, 'product_ids': [16, 59, 83, 96]}, 0: {'customer_id': 0, 'orders': 2, 'product_ids': [30, 47]}, 1: {'customer_id': 1, 'orders': 2, 'product_ids': [19, 29]}, 6: {'customer_id': 6, 'orders': 2, 'product_ids': [3, 84]}, 8: {'customer_id': 8, 'orders': 1, 'product_ids': [8]}, 2: {'customer_id': 2, 'orders': 2, 'product_ids': [23, 37]}, 4: {'customer_id': 4, 'orders': 1, 'product_ids': [84]}, 9: {'customer_id': 9, 'orders': 3, 'product_ids': [19, 35, 95]}}
{5: {'customer_id': 5, 'orders': 1, 'product_ids': [75]}, 7: {'customer_id': 7, 'orders': 2, 'product_ids': [35, 44]}, 3: {'customer_id': 3, 'orders': 4, 'product_ids': [16, 59, 83, 96]}, 0: {'customer_id': 0, 'orders': 2, 'product_ids': [30, 47]}, 1: {'customer_id': 1, 'orders': 2, 'product_ids': [19, 29]}, 6: {'customer_id': 6, 'orders': 2, 'product_ids': [3, 84]}, 8: {'customer_id': 8,

In [128]:
import cloudpickle

#

gen = OrderGenerator()


#

built_tn.reset()
source_m_list = []
sink_m_list = []
for _ in range(10):
    m = gen.generate()
    source_m_list += [m]
    built_tn.push(source_str, [m])
    sink_m_list += built_tn.latest()[sink_str]

#

x = cloudpickle.dumps(built_tn._evaluator)

# print(built_tn.latest())

built_tn.reset()

# print(built_tn.latest())

y = cloudpickle.loads(x)

built_tn._evaluator = y

# print(built_tn.latest())

#

for _ in range(10):
    m = gen.generate()
    source_m_list += [m]
    built_tn.push(source_str, [m])
    sink_m_list += built_tn.latest()[sink_str]

#

source_key_int_value_dict_dict = {}
for m in source_m_list:
    order_id_int = m["value"]["id"]
    product_id_int = m["value"]["product_id"]
    customer_id_int = m["value"]["customer_id"]
    #
    agg_orders_int = source_key_int_value_dict_dict.get(customer_id_int, {}).get("orders", 0)
    agg_product_ids_int_list = source_key_int_value_dict_dict.get(customer_id_int, {}).get("product_ids", [])
    #
    source_key_int_value_dict_dict[customer_id_int] = {"customer_id": customer_id_int,
                                                       "orders": agg_orders_int + 1,
                                                       "product_ids": sorted(agg_product_ids_int_list + [product_id_int])}

#

sink_key_int_value_dict_dict = {}
for m in sink_m_list:
    sink_key_int_value_dict_dict[m["key"]] = m["value"]

#

print(source_key_int_value_dict_dict)
print(sink_key_int_value_dict_dict)

if source_key_int_value_dict_dict != sink_key_int_value_dict_dict:
    raise Exception("Test failed.")

#

print("Test successful.")


{0: {'customer_id': 0, 'orders': 3, 'product_ids': [12, 54, 98]}, 4: {'customer_id': 4, 'orders': 2, 'product_ids': [90, 90]}, 5: {'customer_id': 5, 'orders': 4, 'product_ids': [38, 55, 66, 66]}, 8: {'customer_id': 8, 'orders': 2, 'product_ids': [42, 76]}, 9: {'customer_id': 9, 'orders': 3, 'product_ids': [16, 16, 92]}, 2: {'customer_id': 2, 'orders': 2, 'product_ids': [35, 80]}, 3: {'customer_id': 3, 'orders': 1, 'product_ids': [91]}, 7: {'customer_id': 7, 'orders': 2, 'product_ids': [16, 67]}, 6: {'customer_id': 6, 'orders': 1, 'product_ids': [61]}}
{0: {'customer_id': 0, 'orders': 3, 'product_ids': [12, 54, 98]}, 4: {'customer_id': 4, 'orders': 2, 'product_ids': [90, 90]}, 5: {'customer_id': 5, 'orders': 4, 'product_ids': [38, 55, 66, 66]}, 8: {'customer_id': 8, 'orders': 2, 'product_ids': [42, 76]}, 9: {'customer_id': 9, 'orders': 3, 'product_ids': [16, 16, 92]}, 2: {'customer_id': 2, 'orders': 2, 'product_ids': [35, 80]}, 3: {'customer_id': 3, 'orders': 1, 'product_ids': [91]}, 7: